# Prompt-05: Mid-long Term Trend Analysis

## Environment
- Kernel: **Python 3.10 (py310)**
- Project convention: all outputs go to `outputs/ch05_midlong_term_trend/`
- Data source: `outputs/ch01_data_preprocessing/ch01_cleaned_data.csv`

## Reference
- Project utility modules (`utils.config`, `utils.data_loader`, `utils.output_manager`) are imported from `src/`.
- Follow the project convention for path configuration, plotting style, and artifact saving.

In [ ]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Path setup (ensure utils can be imported)
SCRIPT_DIR = os.path.dirname(os.path.abspath('__file__'))
SRC_DIR = os.path.dirname(SCRIPT_DIR)
PROJECT_ROOT = os.path.dirname(SRC_DIR)
sys.path.insert(0, SRC_DIR)

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import STL

# Import project utilities
from utils.config import CITIES, OUTPUT_BASE, PLT_STYLE
from utils.data_loader import load_preprocessed
from utils.output_manager import save_dataframe, save_figure, ensure_dir

# === Chinese font config ===
plt.rcParams.update({
    'font.sans-serif': ['DejaVu Sans', 'SimHei', 'Noto Sans CJK SC', 'WenQuanYi Micro Hei'],
    'axes.unicode_minus': False,
    'figure.dpi': 150,
    'savefig.dpi': 150,
    'font.size': 12,
})

# === Global config ===
INPUT_FILE = os.path.join(PROJECT_ROOT, 'outputs', 'ch01_data_preprocessing', 'ch01_cleaned_data.csv')
OUTPUT_DIR = os.path.join(OUTPUT_BASE, 'ch05_midlong_term_trend')
CITY_LIST = ['Laayoune', 'Boujdour', 'Foum eloued', 'Marrakech']
STL_PERIOD = 12
STL_MIN_OBS = 2 * STL_PERIOD  # 24

# Color scheme
COLORS = {
    'Laayoune': 'steelblue',
    'Boujdour': 'darkorange',
    'Foum eloued': 'green',
    'Marrakech': 'red'
}

# Quarter mapping
QUARTER_MAP = {
    12: 'Q1(Winter)', 1: 'Q1(Winter)', 2: 'Q1(Winter)',
    3: 'Q2(Spring)',  4: 'Q2(Spring)',  5: 'Q2(Spring)',
    6: 'Q3(Summer)',  7: 'Q3(Summer)',  8: 'Q3(Summer)',
    9: 'Q4(Autumn)', 10: 'Q4(Autumn)', 11: 'Q4(Autumn)'
}
QUARTER_ORDER = ['Q1(Winter)', 'Q2(Spring)', 'Q3(Summer)', 'Q4(Autumn)']

# Ensure output directory exists
ensure_dir(OUTPUT_DIR)

print(f'Input:  {INPUT_FILE}')
print(f'Output: {OUTPUT_DIR}')

## Step 5.1: Multi-granularity Resampling

Aggregate the 10-minute high-frequency data into **daily / monthly / quarterly** average load.

- Equal-weight mean across all zone columns (primary metric)
- Sum across zones (secondary metric)
- NaN handling: forward-fill + back-fill
- Save the dual-metric monthly summary to CSV.

In [ ]:
# Step 5.1: Multi-granularity resampling

print('=' * 60)
print('Step 5.1: Multi-granularity Resampling')
print('=' * 60)

# Load data
print('\nLoading data...')
df = load_preprocessed(INPUT_FILE)
print(f"Loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Time range: {df.index.min()} ~ {df.index.max()}")

zone_cols = [c for c in df.columns if c.startswith('zone')]
print(f"Zone columns: {zone_cols}")
print(f"Cities: {df['city'].unique()}")

resampled_data = {}
monthly_avg_dict = {}
monthly_sum_dict = {}

for city in CITY_LIST:
    city_df = df[df['city'] == city].copy()

    # Equal-weight mean (primary metric)
    city_df['total_load'] = city_df[zone_cols].mean(axis=1)
    # Sum (secondary metric)
    city_df['total_sum'] = city_df[zone_cols].sum(axis=1)

    # Multi-granularity aggregation
    daily = city_df['total_load'].resample('D').mean()
    monthly = city_df['total_load'].resample('ME').mean()
    quarterly = city_df['total_load'].resample('QE').mean()

    # Sum-based monthly
    monthly_sum = city_df['total_sum'].resample('ME').mean()

    # NaN check & handling
    for freq_name, freq_series in [('daily', daily), ('monthly', monthly), ('quarterly', quarterly)]:
        nan_count = freq_series.isnull().sum()
        total_count = len(freq_series)
        if nan_count > 0:
            nan_ratio = nan_count / total_count * 100 if total_count > 0 else 0
            print(f"  [WARNING] {city} {freq_name}: {nan_count}/{total_count} NaN ({nan_ratio:.1f}%), using forward fill")
            freq_series = freq_series.ffill().bfill()
            if freq_name == 'daily':
                daily = freq_series
            elif freq_name == 'monthly':
                monthly = freq_series
            else:
                quarterly = freq_series
        else:
            print(f"  {city} {freq_name}: no NaN, data complete")

    resampled_data[city] = {
        'daily': daily,
        'monthly': monthly,
        'quarterly': quarterly
    }
    monthly_avg_dict[city] = monthly
    monthly_sum_dict[city] = monthly_sum

    print(f"{city}: daily {len(daily)} rows ({daily.index.min().date()}~{daily.index.max().date()}), "
          f"monthly {len(monthly)} rows, quarterly {len(quarterly)} rows")
    print(f"  Daily load range: {daily.min():.1f}~{daily.max():.1f} kW, mean: {daily.mean():.1f} kW")

# Save monthly summary (dual metric)
all_monthly_avg = pd.DataFrame()
all_monthly_sum = pd.DataFrame()
for city in CITY_LIST:
    all_monthly_avg = pd.concat([all_monthly_avg, monthly_avg_dict[city].to_frame(f'{city}_monthly_avg_kw')], axis=1)
    all_monthly_sum = pd.concat([all_monthly_sum, monthly_sum_dict[city].to_frame(f'{city}_monthly_sum_kw')], axis=1)

all_monthly = pd.concat([all_monthly_avg, all_monthly_sum], axis=1)
save_dataframe(all_monthly, 'ch05_daily_monthly_quarterly.csv', OUTPUT_DIR)
print(f"\nMonthly summary: {all_monthly.shape}, dual metric (equal-weight mean + sum)")

## Step 5.2: STL Time Series Decomposition (with MA fallback)

Decompose each city's monthly load series into **Trend + Seasonal + Residual**.

- **Normal path**: `statsmodels.tsa.seasonal.STL` with `period=12`, `robust=True`
- **Fallback path** (when data < 24 months): 3-month centered moving average for trend, monthly historical mean deviation for seasonal
- Save decomposition plots and component CSVs per city.

In [ ]:
# Step 5.2: STL decomposition (with MA fallback)

print('=' * 60)
print('Step 5.2: STL Time Series Decomposition')
print('=' * 60)

stl_results = {}

for city in CITY_LIST:
    monthly = resampled_data[city]['monthly']
    n_obs = len(monthly.dropna())
    print(f"\n{city}: monthly data {n_obs} months")

    if n_obs >= STL_MIN_OBS:
        # Normal STL decomposition
        method = 'STL'
        stl = STL(monthly, period=STL_PERIOD, robust=True)
        result = stl.fit()
        trend = result.trend
        seasonal = result.seasonal
        residual = result.resid
        observed = result.observed
        print(f"  Method: STL (period={STL_PERIOD}, robust=True)")
        print(f"  Trend range: {trend.min():.1f}~{trend.max():.1f} kW")
        print(f"  Seasonal amplitude: {seasonal.max() - seasonal.min():.1f} kW")
        print(f"  Residual std: {residual.std():.1f} kW")
    else:
        # Fallback: moving average decomposition
        method = 'Moving Average'
        print(f"  [WARNING] {city}: only {n_obs} monthly data points (<{STL_MIN_OBS}), using moving average instead of STL")

        observed = monthly.copy()
        # Trend: 3-month centered moving average
        trend = observed.rolling(window=3, center=True).mean()
        trend = trend.bfill().ffill()
        # Seasonal: monthly historical mean - overall mean
        monthly_mean = observed.mean()
        seasonal_pattern = observed.groupby(observed.index.month).mean() - monthly_mean
        seasonal = observed.index.month.map(seasonal_pattern)
        seasonal.index = observed.index
        # Residual
        residual = observed - trend - seasonal
        print(f"  Method: 3-month Centered Moving Average (fallback)")
        print(f"  Trend range: {trend.min():.1f}~{trend.max():.1f} kW")

    stl_results[city] = {
        'observed': observed,
        'trend': trend,
        'seasonal': seasonal,
        'residual': residual,
        'method': method
    }

    # Plot decomposition
    fig, axes = plt.subplots(4, 1, figsize=(14, 12), dpi=150, sharex=True)

    method_label = f'STL (period={STL_PERIOD}, robust=True)' if method == 'STL' else 'Moving Average (3M center)'
    title_suffix = '' if method == 'STL' else ' [MA Fallback - Data < 24 months]'

    axes[0].plot(observed, label='Observed', color='steelblue', linewidth=1.5)
    axes[0].set_title(f'{city} - Time Series Decomposition ({method_label}){title_suffix}',
                      fontsize=14, fontweight='bold')
    axes[0].legend(loc='upper right', fontsize=10)
    axes[0].grid(True, alpha=0.3)
    axes[0].set_ylabel('Load (kW)', fontsize=11)

    axes[1].plot(trend, label='Trend', color='darkorange', linewidth=2)
    axes[1].legend(loc='upper right', fontsize=10)
    axes[1].grid(True, alpha=0.3)
    axes[1].set_ylabel('Load (kW)', fontsize=11)

    axes[2].plot(seasonal, label='Seasonal', color='green', linewidth=1.5)
    axes[2].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    axes[2].legend(loc='upper right', fontsize=10)
    axes[2].grid(True, alpha=0.3)
    axes[2].set_ylabel('Load (kW)', fontsize=11)

    axes[3].plot(residual, label='Residual', color='red', alpha=0.7, linewidth=1)
    axes[3].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    axes[3].legend(loc='upper right', fontsize=10)
    axes[3].grid(True, alpha=0.3)
    axes[3].set_xlabel('Time', fontsize=12)
    axes[3].set_ylabel('Load (kW)', fontsize=11)

    plt.tight_layout()
    city_key = city.lower().replace(' ', '_')
    save_figure(fig, f'ch05_stl_decomposition_{city_key}.png', OUTPUT_DIR)

    # Save decomposition data
    decomp_df = pd.DataFrame({
        'observed': observed,
        'trend': trend,
        'seasonal': seasonal,
        'residual': residual,
        'method': method
    })
    save_dataframe(decomp_df, f'ch05_stl_components_{city_key}.csv', OUTPUT_DIR)

print("\nStep 5.2 complete: all city time series decompositions executed")

## Step 5.3: Trend Extraction & Visualization

- **4-city trend comparison**: overlay decomposed trend components on a single chart
- **Per-city trend fit**: monthly observed load vs. trend component with shaded seasonal+residual band

In [ ]:
# Step 5.3: Trend visualization

print('=' * 60)
print('Step 5.3: Trend Extraction & Visualization')
print('=' * 60)

# 4-city trend comparison
fig, ax = plt.subplots(figsize=(14, 6), dpi=150)

for city in CITY_LIST:
    r = stl_results[city]
    trend = r['trend']
    trend_start = trend.iloc[0]
    trend_end = trend.iloc[-1]
    trend_change = trend_end - trend_start
    trend_pct = (trend_change / trend_start) * 100 if trend_start != 0 else 0
    direction = "Up" if trend_change > 0 else ("Down" if trend_change < 0 else "Flat")
    print(f"  {city}: Trend {direction} ({trend_pct:+.1f}%), {trend_start:.1f} -> {trend_end:.1f} kW [{r['method']}]")

    ax.plot(trend.index, trend, label=f'{city} ({r["method"]})',
            color=COLORS.get(city, 'gray'), linewidth=2)

ax.set_title('4-City Load Long-term Trend Comparison (Decomposed Trend Component)',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Time', fontsize=12)
ax.set_ylabel('Trend Load (kW)', fontsize=12)
ax.legend(fontsize=10, loc='best')
ax.grid(True, alpha=0.3)
plt.tight_layout()
save_figure(fig, 'ch05_trend_component.png', OUTPUT_DIR)

# Per-city trend fit
for city in CITY_LIST:
    r = stl_results[city]
    fig, ax = plt.subplots(figsize=(14, 5), dpi=150)
    ax.plot(r['observed'].index, r['observed'], label='Monthly Avg Load',
            color='steelblue', alpha=0.5, linewidth=1)
    ax.plot(r['trend'].index, r['trend'], label=f'Trend ({r["method"]})',
            color='darkorange', linewidth=2.5)
    ax.fill_between(r['observed'].index, r['observed'], r['trend'],
                    alpha=0.15, color='gray', label='Seasonal + Residual')

    method_note = '' if r['method'] == 'STL' else ' [MA Fallback]'
    ax.set_title(f'{city} - Monthly Load vs Trend Component{method_note}',
                 fontsize=14, fontweight='bold')
    ax.set_xlabel('Time', fontsize=12)
    ax.set_ylabel('Load (kW)', fontsize=12)
    ax.legend(fontsize=10, loc='best')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    city_key = city.lower().replace(' ', '_')
    save_figure(fig, f'ch05_trend_fit_{city_key}.png', OUTPUT_DIR)

print("\nStep 5.3 complete: trend visualization generated")

## Step 5.4: Seasonal Strength Calculation

Compute the seasonal strength index:

$$F_s = \max\left(0,\; 1 - \frac{\text{Var}(R)}{\text{Var}(S) + \text{Var}(R)}\right)$$

- $F_s > 0.7$: **Strong** seasonality
- $0.4 < F_s \leq 0.7$: **Moderate** seasonality
- $F_s \leq 0.4$: **Weak** seasonality

Save results to CSV and generate a horizontal bar chart.

In [ ]:
# Step 5.4: Seasonal strength calculation

print('=' * 60)
print('Step 5.4: Seasonal Strength Calculation')
print('=' * 60)

records = []

for city in CITY_LIST:
    r = stl_results[city]
    var_s = r['seasonal'].var()
    var_r = r['residual'].var()

    f_s = max(0, 1 - var_r / (var_s + var_r)) if (var_s + var_r) > 0 else 0.0

    if f_s > 0.7:
        level = 'Strong'
    elif f_s > 0.4:
        level = 'Moderate'
    else:
        level = 'Weak'

    # Mark fallback cities
    if r['method'] != 'STL':
        level += ' (MA fallback)'

    records.append({
        'city': city,
        'var_seasonal': round(var_s, 4),
        'var_residual': round(var_r, 4),
        'seasonal_strength': round(f_s, 4),
        'strength_level': level,
        'decomposition_method': r['method']
    })
    print(f"  {city}: F_s = {f_s:.4f} ({level}), Var(S)={var_s:.2f}, Var(R)={var_r:.2f} [{r['method']}]")

strength_df = pd.DataFrame(records)
save_dataframe(strength_df, 'ch05_seasonal_strength.csv', OUTPUT_DIR, index=False)

# Visualization: seasonal strength comparison bar chart
fig, ax = plt.subplots(figsize=(10, 5), dpi=150)
strength_sorted = strength_df.sort_values('seasonal_strength', ascending=True)
bar_colors = ['#e74c3c' if v > 0.7 else ('#f39c12' if v > 0.4 else '#27ae60')
              for v in strength_sorted['seasonal_strength']]

bars = ax.barh(strength_sorted['city'], strength_sorted['seasonal_strength'],
               color=bar_colors, edgecolor='white', height=0.6)

for bar, val in zip(bars, strength_sorted['seasonal_strength']):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}', va='center', fontsize=11, fontweight='bold')

ax.set_xlabel('Seasonal Strength F_s', fontsize=12)
ax.set_title('4-City Load Seasonal Strength Comparison', fontsize=14, fontweight='bold')
ax.set_xlim(0, 1.1)
ax.axvline(x=0.7, color='red', linestyle='--', alpha=0.5, label='Strong threshold (0.7)')
ax.axvline(x=0.4, color='orange', linestyle='--', alpha=0.5, label='Moderate threshold (0.4)')
ax.legend(fontsize=10, loc='lower right')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
save_figure(fig, 'ch05_seasonal_strength.png', OUTPUT_DIR)

print("\nStep 5.4 complete: seasonal strength calculation done")

## Step 5.5: Quarterly Load Boxplot

For each city, plot daily load distributions grouped by quarter:

- Q1 (Winter): Dec, Jan, Feb
- Q2 (Spring): Mar, Apr, May
- Q3 (Summer): Jun, Jul, Aug
- Q4 (Autumn): Sep, Oct, Nov

Sahara cities (Laayoune, Boujdour, Foum eloued) are annotated for potential extreme-heat load patterns.

In [ ]:
# Step 5.5: Quarterly load boxplot

print('=' * 60)
print('Step 5.5: Quarterly Load Boxplot')
print('=' * 60)

sahara_cities = ['Laayoune', 'Boujdour', 'Foum eloued']

for city in CITY_LIST:
    daily = resampled_data[city]['daily'].to_frame(name='load_kw')
    daily['quarter'] = daily.index.month.map(QUARTER_MAP)

    # Statistics summary
    print(f"\n  {city} Quarterly Load Statistics:")
    for q in QUARTER_ORDER:
        q_data = daily[daily['quarter'] == q]['load_kw']
        if len(q_data) > 0:
            print(f"    {q}: n={len(q_data)}, mean={q_data.mean():.1f}, "
                  f"median={q_data.median():.1f}, std={q_data.std():.1f}, "
                  f"range=[{q_data.min():.1f}, {q_data.max():.1f}]")

    # Plot boxplot
    fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
    sns.boxplot(data=daily, x='quarter', y='load_kw', order=QUARTER_ORDER,
                palette='Set2', ax=ax, width=0.6,
                flierprops={'marker': 'o', 'markersize': 4, 'alpha': 0.5})
    sns.stripplot(data=daily, x='quarter', y='load_kw', order=QUARTER_ORDER,
                  color='black', alpha=0.3, size=3, ax=ax, jitter=True)

    subtitle = ''
    if city in sahara_cities:
        subtitle = '\n(Note: Sahara city - extreme heat may cause unique load patterns)'

    ax.set_title(f'{city} - Quarterly Load Distribution (Daily Data){subtitle}',
                 fontsize=14, fontweight='bold')
    ax.set_xlabel('Quarter', fontsize=12)
    ax.set_ylabel('Daily Avg Load (kW)', fontsize=12)
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    city_key = city.lower().replace(' ', '_')
    save_figure(fig, f'ch05_quarterly_boxplot_{city_key}.png', OUTPUT_DIR)

print("\nStep 5.5 complete: quarterly load boxplots generated")

## Step 5.6: Monthly Load Heatmap

Generate a **month x year** heatmap of monthly average load for each city.

- Rows: months 1-12
- Columns: years
- Color: `YlOrRd` colormap with annotated values

In [ ]:
# Step 5.6: Monthly load heatmap

print('=' * 60)
print('Step 5.6: Monthly Load Heatmap')
print('=' * 60)

for city in CITY_LIST:
    monthly = resampled_data[city]['monthly'].to_frame(name='load_kw')
    monthly['year'] = monthly.index.year
    monthly['month'] = monthly.index.month

    # Build pivot table
    pivot = monthly.pivot_table(values='load_kw', index='month', columns='year', aggfunc='mean')
    pivot.index = [f'{m}' for m in pivot.index]

    print(f"\n  {city}: years={pivot.columns.tolist()}, months={pivot.index.tolist()}, "
          f"missing={pivot.isnull().sum().sum()}")
    if pivot.notna().any().any():
        print(f"    min={pivot.min().min():.1f} kW, max={pivot.max().max():.1f} kW")

    # Plot heatmap
    fig, ax = plt.subplots(figsize=(10, 7), dpi=150)
    sns.heatmap(pivot, ax=ax, cmap='YlOrRd', annot=True, fmt='.1f',
                linewidths=0.5, linecolor='white',
                cbar_kws={'label': 'Monthly Avg Load (kW)', 'shrink': 0.8})

    ax.set_title(f'{city} - Monthly Load Heatmap (kW)', fontsize=14, fontweight='bold')
    ax.set_xlabel('Year', fontsize=12)
    ax.set_ylabel('Month', fontsize=12)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

    plt.tight_layout()
    city_key = city.lower().replace(' ', '_')
    save_figure(fig, f'ch05_monthly_heatmap_{city_key}.png', OUTPUT_DIR)

print("\nStep 5.6 complete: monthly load heatmaps generated")

## Step 5.7: Year-over-Year (YoY) / Month-over-Month (MoM) Analysis

Calculate percentage change rates:

- **MoM**: `pct_change(periods=1)` -- month-over-month change
- **YoY**: `pct_change(periods=12)` -- year-over-year change

Save the combined table and generate line charts for both YoY and MoM rates.

In [ ]:
# Step 5.7: YoY / MoM analysis

print('=' * 60)
print('Step 5.7: YoY / MoM Analysis')
print('=' * 60)

yoy_mom_records = []

for city in CITY_LIST:
    monthly = resampled_data[city]['monthly']

    mom = monthly.pct_change(periods=1) * 100   # MoM (%)
    yoy = monthly.pct_change(periods=12) * 100  # YoY (%)

    temp = pd.DataFrame({
        'monthly_avg_kw': monthly,
        'mom_change_pct': mom,
        'yoy_change_pct': yoy
    })
    temp['city'] = city
    yoy_mom_records.append(temp)

    n_months = len(monthly)
    n_yoy_valid = yoy.dropna().shape[0]
    print(f"\n  {'=' * 50}")
    print(f"  {city} - YoY / MoM Analysis")
    print(f"  {'=' * 50}")
    print(f"    Data range: {monthly.index.min().date()} ~ {monthly.index.max().date()} ({n_months} months)")
    print(f"    MoM NaN: {mom.isnull().sum()} (1st month has no MoM)")
    print(f"    YoY NaN: {yoy.isnull().sum()} (first 12 months have no YoY)")

    if n_months < 13:
        print(f"    [WARNING] {city}: Only {n_months} months of data, YoY analysis has no statistical significance")

    recent = temp[['monthly_avg_kw', 'mom_change_pct', 'yoy_change_pct']].tail(6)
    print(f"\n    Last 6 months:")
    print(recent.to_string(float_format=lambda x: f'{x:.1f}' if pd.notna(x) else 'NaN'))

    valid_mom = mom.dropna()
    if len(valid_mom) > 0:
        print(f"\n    MoM stats: mean={valid_mom.mean():.2f}%, std={valid_mom.std():.2f}%, "
              f"range=[{valid_mom.min():.2f}%, {valid_mom.max():.2f}%]")

    valid_yoy = yoy.dropna()
    if len(valid_yoy) > 0:
        print(f"    YoY stats: mean={valid_yoy.mean():.2f}%, std={valid_yoy.std():.2f}%, "
              f"range=[{valid_yoy.min():.2f}%, {valid_yoy.max():.2f}%]")

# Save summary table
yoy_mom_df = pd.concat(yoy_mom_records)
save_dataframe(yoy_mom_df, 'ch05_yoy_mom_analysis.csv', OUTPUT_DIR)

# YoY change rate chart
fig, ax = plt.subplots(figsize=(14, 6), dpi=150)
for city in CITY_LIST:
    city_data = yoy_mom_df[yoy_mom_df['city'] == city]
    valid = city_data['yoy_change_pct'].dropna()
    if len(valid) > 0:
        ax.plot(city_data.index, city_data['yoy_change_pct'],
                label=city, color=COLORS.get(city, 'gray'),
                linewidth=1.5, marker='o', markersize=4)
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.8, alpha=0.5)
ax.set_title('4-City Monthly Load YoY Change Rate (%)', fontsize=14, fontweight='bold')
ax.set_xlabel('Time', fontsize=12)
ax.set_ylabel('YoY Change Rate (%)', fontsize=12)
ax.legend(fontsize=10, loc='best')
ax.grid(True, alpha=0.3)
plt.tight_layout()
save_figure(fig, 'ch05_yoy_change_rate.png', OUTPUT_DIR)

# MoM change rate chart
fig, ax = plt.subplots(figsize=(14, 6), dpi=150)
for city in CITY_LIST:
    city_data = yoy_mom_df[yoy_mom_df['city'] == city]
    ax.plot(city_data.index, city_data['mom_change_pct'],
            label=city, color=COLORS.get(city, 'gray'),
            linewidth=1.5, marker='o', markersize=4)
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.8, alpha=0.5)
ax.set_title('4-City Monthly Load MoM Change Rate (%)', fontsize=14, fontweight='bold')
ax.set_xlabel('Time', fontsize=12)
ax.set_ylabel('MoM Change Rate (%)', fontsize=12)
ax.legend(fontsize=10, loc='best')
ax.grid(True, alpha=0.3)
plt.tight_layout()
save_figure(fig, 'ch05_mom_change_rate.png', OUTPUT_DIR)

print("\nStep 5.7 complete: YoY / MoM analysis done")

In [ ]:
# Summary: list all artifacts

print('=' * 60)
print('Prompt-05 COMPLETE - All artifacts generated')
print('=' * 60)

artifacts = sorted(os.listdir(OUTPUT_DIR))
print(f'\nTotal artifacts in {OUTPUT_DIR}: {len(artifacts)}')
for a in artifacts:
    fpath = os.path.join(OUTPUT_DIR, a)
    size_kb = os.path.getsize(fpath) / 1024
    print(f'  {a} ({size_kb:.1f} KB)')